<a href="https://colab.research.google.com/github/PacktPublishing/Building-AI-Agents-for-Finance-/blob/main/Chapter%201/chapter_1_lab_3_non-agentic_vs_agentic-workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 1 - Lab 3: <font color='blue'>Agentic vs non-Agentic workflow</font>

**<font color='purple'>Goal</font>**:
In this lab, you'll compare **Agentic** and **non-agentiw workflow**:

* Chatting with an LLM without access to external resources
* Chatting with an LLM with augmented information using external APIs:
    -  Fetch Historical Prices for a given stock (The same use case in the book)
    -  Collect News (Additional use case to practice)

**<font color='purple'>Tech stack</font>**:

We'll use :
* OpenAI Responses API ==> Standard calls + Function Callings
* OpenAI Agent SDK ==> Simple Agent Abstraction
* NewsApi: https://newsapi.org/

You need to create API Keys in both OpenAI (you need to add some credits) and NewsAPI (free until a certain # of calls).

Knowledge Cutoff of GPT-4.1 is June 2024

## Install packages

Install all notebook dependencies before importing OpenAI. Keeping them in one transaction avoids loading an older `typing_extensions` before the Agents SDK upgrades it. If this cell upgrades packages in an existing kernel, restart the kernel once and then choose **Run All**.

In [9]:
!uv pip install --python {__import__('sys').executable} --upgrade -q openai openai-agents newsapi-python yfinance "typing-extensions>=4.15"

Add you OpenAI Key in Google colab: in the left vertical menu, you have an icon key. Click on it, and add your OpenAI key with a name

In [10]:
import os

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    NEWS_API_KEY = userdata.get('NEWS_API_KEY')
except ImportError:
    from dotenv import find_dotenv, load_dotenv

    # Search the working directory and its parents for the local .env file.
    load_dotenv(find_dotenv(usecwd=True), override=False)
    OPENAI_API_KEY = os.environ['OPENAI_API_KEY']
    NEWS_API_KEY = os.environ['NEWS_API_KEY']

To have more information on how to call the API, you can read the documentation below.

In the various labs, you'll have a detailed overview on how to use the API.

https://platform.openai.com/docs/api-reference



# 1- Non Agentic Workflow

## Historical Price

### Responses API - GPT-4.1

https://platform.openai.com/docs/api-reference/responses

In [20]:
from openai import OpenAI

client = OpenAI(api_key = OPENAI_API_KEY)

response = client.responses.create(
  model="gpt-4.1",
  input="What is the last price of NVIDIA."
)

print(response)

Response(id='resp_0271f05fc7d03197006ae107d0275087d1b2bd79ed10ab5596', access_programs=None, created_at=1793132496.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_0271f05fc7d03197006ae107d0a20c87d1ae06912e248c54d1', content=[ResponseOutputText(annotations=[], text='As of the latest available data, NVIDIA Corporation (ticker symbol: NVDA) closed at **$126.57** per share on June 14, 2024.\n\nPlease note that stock prices are subject to market fluctuations and may have changed since this information was retrieved. For real-time prices, consider checking a financial news website or your brokerage platform.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1793132497.0, conversation=None, max_output_tokens=None, max_

In [21]:
print(response.output[0].content[0].text)

As of the latest available data, NVIDIA Corporation (ticker symbol: NVDA) closed at **$126.57** per share on June 14, 2024.

Please note that stock prices are subject to market fluctuations and may have changed since this information was retrieved. For real-time prices, consider checking a financial news website or your brokerage platform.


### Results

As yu can see, the LLM was not able to provide the last price of NVIDIA, as its knwoledge cufoff was on June 2024.

### Another Try

In this part, I'll provide a web site link. let's see what happens:

In [14]:
from openai import OpenAI

client = OpenAI(api_key = OPENAI_API_KEY)

response2 = client.responses.create(
  model="gpt-4.1",
  input="What is the last price of NVIDIA. Can you access to https://finance.yahoo.com/quote/NVDA to collect the last price? "
)

print(response2.output[0].content[0].text)

I can't browse the internet in real time or access external websites like Yahoo Finance directly. However, you can easily check the latest price of NVIDIA (NVDA) stock by visiting this link: [https://finance.yahoo.com/quote/NVDA](https://finance.yahoo.com/quote/NVDA).

If you would like, you can tell me the current price you see, and I can help you analyze it or provide more context or information! Alternatively, I can provide guidance on using APIs or Python code to get real-time stock prices if you're interested.


I have the same issue, the LLM cannot provide the price because it cannot access and browse external websites like this.

## News

### Responses API

In [15]:
from openai import OpenAI

client = OpenAI(api_key = OPENAI_API_KEY)

response = client.responses.create(
  model="gpt-4.1",
  input="Give me the latest news about NVIDIA."
)

print(response.output[0].content[0].text)

As of June 2024, here are some of the latest news highlights regarding **NVIDIA**:

---

### 1. **Record-Breaking Earnings and Market Cap**
- **NVIDIA reported a record-breaking Q1 FY2025 revenue, beating Wall Street expectations** due to surging demand for AI chips, particularly the H100 and newly announced Blackwell GPU architecture.
- The company’s stock recently **passed $3 trillion in market value**, briefly overtaking Apple to become the world’s second-most valuable publicly traded company after Microsoft.

---

### 2. **AI and Data Center Dominance**
- **NVIDIA continues to dominate the data center GPU market**, with major cloud providers like Microsoft Azure, Google Cloud, and Amazon AWS heavily investing in NVIDIA hardware for AI workloads.
- Its A100 and H100 chips reportedly power the current wave of generative AI models (e.g., ChatGPT, Google Gemini).

---

### 3. **New Product Announcements**
- **NVIDIA officially unveiled the “Blackwell” chip architecture** at its GTC 202

### Results

If you search for "NVIDIA Surpasses $3 Trillion Market Cap" in the web, you'll see that this news is from June 2024... as it's mentionned by the llm itself in the output.

# 2- Agentic Workflow:

In the following implementation, we'll use tools call to avoid knowledge cutoff and therefore build a basic agentic systeme

## Tools Calling without agent abstraction: Historical

Here, you'll learn how to use function calling with Responses API, without using any agent abstraction

1. You need to define what we call "Tools" object: Type (function), name, Description, parameters...
2. Declare your function: Tool to access external API
3. Call Responses API by enabling: "tools"

https://platform.openai.com/docs/guides/function-calling?api-mode=responses

In [16]:
import yfinance as yf

In [24]:
from openai import OpenAI
import json

client = OpenAI(api_key = OPENAI_API_KEY)

# 1. Define a list of callable tools for the model
tools = [
    {
        "type": "function",
        "name": "get_latest_stock_price",
        "description": "Fetch the current stock price for the given symbol.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "A ticker symbol of a given stock",
                },
            },
            "required": ["symbol"],
        },
    },
]

# Define the tool
def get_latest_stock_price(symbol: str) -> dict:
    """
    Fetch the current stock price for the given symbol.
    """

    ticker = yf.Ticker(symbol)
    hist = ticker.history(period="1d")
    latest_close = hist['Close'].iloc[-1]
    latest_date = hist.index[-1].strftime('%Y-%m-%d')

    return {'price': latest_close, 'date': latest_date}


input_list = [
    {"role": "user", "content": "What is the current price of nvidia."}
]

# 2. Prompt the model with tools defined
response = client.responses.create(
    model="gpt-6-luna",
    tools=tools,
    input=input_list,
)

print(response.output)

[ResponseFunctionToolCall(arguments='{"symbol":"NVDA"}', call_id='call_pHCxUuzC7clsg5uwooesYOCP', name='get_latest_stock_price', type='function_call', id='fc_04013524944cb01f006ae108462b4887d1b9bb203b7523f12b', async_=None, caller=None, namespace=None, status='completed')]


In [28]:
print(response.output)

[ResponseFunctionToolCall(arguments='{"symbol":"NVDA"}', call_id='call_pHCxUuzC7clsg5uwooesYOCP', name='get_latest_stock_price', type='function_call', id='fc_04013524944cb01f006ae108462b4887d1b9bb203b7523f12b', async_=None, caller=None, namespace=None, status='completed')]


* This will generate a ResponseFunctionToolCall with the name of the function, and the argument (well interpreted by the model, it gives a symbol instead of the name of the company)
* We then to execute this function with the right argument


In [29]:
# Save function call outputs for subsequent requests
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "get_latest_stock_price":
            print(json.loads(item.arguments))
            # 3. Execute the function logic
            prices = get_latest_stock_price(json.loads(item.arguments)['symbol'])
            print(prices)

            # 4. Provide function call results to the model
            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({
                  "prices": prices
                })
            })

print("Final input:")
print(input_list)

{'symbol': 'NVDA'}
{'price': np.float64(217.94000244140625), 'date': '2026-10-27'}
Final input:
[{'role': 'user', 'content': 'What is the current price of nvidia.'}, ResponseFunctionToolCall(arguments='{"symbol":"NVDA"}', call_id='call_pHCxUuzC7clsg5uwooesYOCP', name='get_latest_stock_price', type='function_call', id='fc_04013524944cb01f006ae108462b4887d1b9bb203b7523f12b', async_=None, caller=None, namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_pHCxUuzC7clsg5uwooesYOCP', 'output': '{"prices": {"price": 217.94000244140625, "date": "2026-10-27"}}'}]


/home/naragon/projects/Building-AI-Agents-for-Finance/.venv/lib/python3.12/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/home/naragon/projects/Building-AI-Agents-for-Finance/.venv/lib/python3.12/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/home/naragon/projects/Building-AI-Agents-for-Finance/.venv/lib/python3.12/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#d

* Send back the whole result to the LLM to interpret and output the final answer

In [ ]:
# Call the API with the updated input list
response = client.responses.create(
    model="gpt-6-luna",
    instructions="Respond only with price and date generated by a tool.",
    tools=tools,
    input=input_list,
)

# 5. The model should be able to give a response!
print("Final output:")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)

Final output:
{
  "id": "resp_0bc728e952ba97f5006adbd2d8eb2087d18d1d6dc7de899156",
  "access_programs": null,
  "created_at": 1792791257.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "Respond only with price and date generated by a tool.",
  "metadata": {},
  "model": "gpt-6-luna",
  "object": "response",
  "output": [
    {
      "id": "msg_0bc728e952ba97f5006adbd2d97ce887d1852f6f0c82294156",
      "content": [
        {
          "annotations": [],
          "text": "The current price of Nvidia (NVDA) is $210.96 as of October 23, 2026.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": null
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "get_latest_stock_price",
      "parameters": {
        "type": "object",
        "properties": {
          "symbol": {
            

https://composio.dev/blog/claude-function-calling-tools

## OpenAI Agent SDK

In this part, we'll be using OpenAI Agent SDK.

You need first to install the package.

We'll create:
*  A tool: to fetch historical prices
*  An agent:
   - Describe the instructions to give to the agent
   - List of the tools the agent can use
   - Model

Verify the OpenAI Agents SDK runtime dependencies

In [12]:
import importlib.metadata
import typing_extensions

typing_extensions_version = importlib.metadata.version("typing-extensions")
if not hasattr(typing_extensions, "sentinel"):
    raise RuntimeError(
        f"typing-extensions {typing_extensions_version} was installed, but this kernel still has "
        "an older copy loaded. Restart the kernel and run the notebook from the top."
    )

print(f"typing-extensions {typing_extensions_version}: compatible")

typing-extensions 4.16.0: compatible


In [13]:
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

#We import the agent components
from agents import Agent, Runner, function_tool

### Historical Prices Agent

#### Tool

➡ Create a tool to fetch historical prices

In [14]:
import yfinance as yf


@function_tool
def get_latest_stock_price(symbol: str) -> dict:
    """
    Fetch the current stock price for the given symbol.
    """

    ticker = yf.Ticker(symbol)
    hist = ticker.history(period="1d") # Fetch the last 1 day of historical data
    if hist.empty:
        raise ValueError(f"No price data returned for {symbol}")
    latest_close = float(hist['Close'].iloc[-1])
    latest_date = hist.index[-1].strftime('%Y-%m-%d')

    return {'price': latest_close, 'date': latest_date}

In [15]:
ticker = yf.Ticker("NVDA")
hist = ticker.history(period="1d")
hist

/Volumes/Zike/MacMini/naragonex/.pyenv/versions/3.13.1/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Volumes/Zike/MacMini/naragonex/.pyenv/versions/3.13.1/lib/python3.13/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/Volumes/Zike/MacMini/naragonex/.pyenv/versions/3.13.1/lib/python3.13/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inpl

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-10-23 00:00:00-04:00,210.940002,213.850006,210.179993,210.960007,79740563,0.0,0.0


#### Agent

##### Define your agent

In [16]:
MODEL = "gpt-4.1"
stock_price_agent = Agent(
    name="Stock Price Agent",
    instructions="Fetch Historical prices for a given stock symbol",
    model = MODEL,
    tools=[get_latest_stock_price],
)

##### Run your agent

Jupyter already runs an event loop, so use the asynchronous `Runner.run` API with top-level `await`. `Runner.run_sync` is intended for regular Python processes where no event loop is active.

In [17]:

output = await Runner.run(
    starting_agent=stock_price_agent,
    input="What is today's date, and what is the current stock price of Nvidia?",
)

print(output.final_output)

/Volumes/Zike/MacMini/naragonex/.pyenv/versions/3.13.1/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Volumes/Zike/MacMini/naragonex/.pyenv/versions/3.13.1/lib/python3.13/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/Volumes/Zike/MacMini/naragonex/.pyenv/versions/3.13.1/lib/python3.13/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inpl

Today's date is October 23, 2026, and the current stock price of Nvidia (NVDA) is $210.96.


### News Agent

In [ ]:
from datetime import date, timedelta
import json

from newsapi import NewsApiClient

newsapi = NewsApiClient(api_key=NEWS_API_KEY)


#### Tool

Here I defined two tools:
* One fetching latest news from NewsAPI ==> **search_news**
* The other summarizing these news using an LLM ==> **summarize_news_api**

Only the ticker is exposed to the agent. The date window is calculated inside the Python function each time it runs. This prevents the model from supplying stale or invalid dates to NewsAPI.


In [ ]:
from openai import OpenAI
client = OpenAI(api_key = OPENAI_API_KEY)

In [ ]:
def _search_news_internal(ticker: str, num_articles: int = 3) -> str:
    """Get the most recent NewsAPI articles for a stock or instrument."""
    # Compute these at call time. Defaults such as ``date.today()`` are otherwise
    # frozen when the cell is executed, and dates exposed as tool arguments can be
    # replaced by the model.
    end_date = date.today()
    start_date = end_date - timedelta(days=7)

    response = newsapi.get_everything(
        q=ticker.strip(),
        from_param=start_date.isoformat(),
        to=end_date.isoformat(),
        language="en",
        sort_by="publishedAt",
        page_size=num_articles,
    )

    articles = response.get("articles", [])
    if not articles:
        return f"No English-language articles found for {ticker!r} in the last 7 days."

    # NewsAPI fields are nullable. In particular, ``content`` is often None, so
    # slicing article["content"] directly can raise an exception inside the tool.
    cleaned_articles = [
        {
            "title": article.get("title") or "",
            "description": article.get("description") or "",
            "published_at": article.get("publishedAt") or "",
            "source": (article.get("source") or {}).get("name") or "",
            "url": article.get("url") or "",
            "content_preview": (article.get("content") or "")[:500],
        }
        for article in articles
    ]
    return json.dumps(cleaned_articles, ensure_ascii=False, indent=2)


@function_tool
def search_news(ticker: str) -> str:
    """Get the latest news for a stock ticker or company name."""
    return _search_news_internal(ticker)


In [ ]:
@function_tool
def summarize_news_api(ticker: str) -> str:
    """Fetch and summarize the latest news for a stock ticker or company name."""
    news = _search_news_internal(ticker)
    response = client.responses.create(
        model="gpt-6-luna",
        input=(
            "Summarize the key insights in these recent news articles. "
            "Include publication dates and source links when available.\n\n"
            f"{news}"
        ),
    )
    return response.output_text


#### Running NewsAPI itself without LLM/Agent

In [ ]:
# Run the same implementation used by the agent. If NewsAPI rejects the request,
# this cell now surfaces the real API error before an LLM can paraphrase it.
news = _search_news_internal("NVDA")
print(news)


In [ ]:
news


In [ ]:
# Another call to verify that the retrieved articles can be summarized.
response = client.responses.create(
    model="gpt-6-luna",
    input=f"Summarize the key insights in these recent news articles:\n\n{news}",
)
print(response.output_text)


In [ ]:
print(response.output[1].content[0].text)

#### Agent: Fetch News

In [ ]:
from agents import Agent, Runner, function_tool

In [ ]:
MODEL = "gpt-6-luna"
news_agent = Agent(
    name="News Agent",
    instructions=(
        "Always use search_news to answer requests for recent company or stock news. "
        "Call it once for each requested company. Report article dates and links, and "
        "clearly say when the tool returns no articles."
    ),
    model=MODEL,
    tools=[search_news],
)


In [ ]:
output = await Runner.run(
    starting_agent=news_agent,
    input="Give me the latest news about Apple.",
)

print(output.final_output)

In [ ]:
output = await Runner.run(
    starting_agent=news_agent,
    input="give me the latest news about Nvidia?",
)

print(output.final_output)

In [ ]:
output = await Runner.run(
    starting_agent=news_agent,
    input="Provide the latest news about Apple and Nvidia.",
)

print(output.final_output)

#### Agent: Summarize news

In [ ]:
MODEL = "gpt-6-luna"
news_agent = Agent(
    name="News Agent",
    instructions=(
        "Always use summarize_news_api to summarize recent company or stock news. "
        "Call it once for each requested company."
    ),
    model=MODEL,
    tools=[summarize_news_api],
)


In [ ]:
output = await Runner.run(
    starting_agent=news_agent,
    input="Summarize the latest news of Apple?",
)

print(output.final_output)

In [ ]:
output = await Runner.run(
    starting_agent=news_agent,
    input="Give me a summary of the latest news of NVIDIA?",
)

print(output.final_output)